In [ ]:
# ============================================
# WEATHER INTELLIGENCE SYSTEM
# ============================================

# STEP 1: IMPORT LIBRARIES
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# ============================================
# STEP 2: LOAD CSV FILE
# ============================================

# Load weather dataset
df = pd.read_csv("weather_data.csv")

print("First 5 Rows:")
print(df.head())

# ============================================
# STEP 3: ETL CLEANING
# ============================================

# Remove missing values
df = df.dropna()

# Standardize column names
df.columns = df.columns.str.lower()

# Example expected columns:
# temperature, humidity, wind_speed

# ============================================
# STEP 4: COMPUTE HEAT INDEX
# ============================================

# Simple heat index formula
df['heat_index'] = (
    df['temperature'] +
    (0.1 * df['humidity']) -
    (0.05 * df['wind_speed'])
)

print("\nDataset After Cleaning:")
print(df.head())

# ============================================
# STEP 5: STORE DATA IN SQLITE
# ============================================

conn = sqlite3.connect(":memory:")

# Store table
df.to_sql("weather", conn, if_exists="replace", index=False)

print("\nData stored in SQLite successfully.")

# Query Example
query = """
SELECT AVG(temperature) as avg_temp,
       AVG(humidity) as avg_humidity
FROM weather
"""

result = pd.read_sql(query, conn)

print("\nWeather Insights:")
print(result)

# ============================================
# STEP 6: DATA VISUALIZATION
# ============================================

# Temperature Chart
plt.figure(figsize=(8,5))
plt.plot(df['temperature'])
plt.title("Temperature Chart")
plt.xlabel("Records")
plt.ylabel("Temperature")
plt.show()

# Humidity Chart
plt.figure(figsize=(8,5))
plt.plot(df['humidity'])
plt.title("Humidity Chart")
plt.xlabel("Records")
plt.ylabel("Humidity")
plt.show()

# Wind Speed Chart
plt.figure(figsize=(8,5))
plt.plot(df['wind_speed'])
plt.title("Wind Speed Chart")
plt.xlabel("Records")
plt.ylabel("Wind Speed")
plt.show()

# ============================================
# STEP 7: MACHINE LEARNING PREDICTION
# ============================================

# Features
X = df[['temperature', 'wind_speed']]

# Target
y = df['humidity']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
predictions = model.predict(X_test)

# Accuracy
mse = mean_squared_error(y_test, predictions)

print("\nModel Mean Squared Error:", mse)

# Sample prediction
sample = [[30, 10]]   # temperature=30, wind_speed=10
predicted_humidity = model.predict(sample)

print("\nPredicted Humidity:", predicted_humidity[0])

# ============================================
# STEP 8: CLOSE DATABASE
# ============================================

conn.close()

print("\nProject Completed Successfully!")